# 🌐 Example 1: Basic Globe (Topography Only)

Welcome! In this tutorial, we will create a simple, single-color 3D globe with surface topography. This is the perfect starting point to verify your installation and make your first 3D print.

### 🌎 Scientific & Design Context
On a true-scale 80 mm desktop globe, Earth's highest mountains and deepest ocean trenches would project outward or inward by less than 0.1 mm—making the globe feel completely smooth. To make mountains and subduction zones tactile, we apply a **vertical exaggeration**. In this notebook, we'll scale the topography by **40×** so the geological features can be easily felt by hand.

## Step 1: Import the Library

We import `globe3d`'s object-oriented components along with standard scientific and plotting tools.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    calculate_displacement_scale
)

## Step 2: Generate the Base Sphere

We define the physical size of our globe in millimeters (standard for 3D printers). We will use a radius of 40 mm (giving an 80 mm diameter globe).

For this demo, we generate **5,000 points** using a golden-ratio Fibonacci spiral so the code runs instantly. *For high-resolution prints, we recommend using 50,000 to 150,000 points.*

In [ ]:
model_radius_mm = 40.0
n_points = 5000

# Create a GlobeModel using the unified constructor
model = GlobeModel(method='fibonacci', n_points=n_points, radius=model_radius_mm)
print(f"Generated sphere with {model.outer.vertices.shape[0]} vertices and {model.outer.faces.shape[0]} faces.")

## Step 3: Load the Topography Dataset (ETOPO)

We load global elevation data from the ETOPO database. Because planetary datasets are extremely high resolution, we downsample the grid (selecting every 10th row and column) so that calculations are fast and responsive inside this notebook.

In [ ]:
netcdf_path = "../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc"

# Load latitude, longitude, and data grid arrays
full_grid = GeographicGrid.from_netcdf(netcdf_path, lat_var='lat', lon_var='lon', data_var='z')

# Downsample the grid for faster interpolation in Python
grid_ds = GeographicGrid(
    lats=full_grid.lats[::10],
    lons=full_grid.lons[::10],
    grid=full_grid.grid[::10, ::10]
)
print(f"Downsampled grid shape: {grid_ds.grid.shape}")

## Step 4: Calculate the Displacement Scale Factor

The ETOPO topography grid represents elevations in **meters**, but our 3D model is in **millimeters**. We use the `calculate_displacement_scale` helper to convert units automatically and scale the values by our 40× vertical exaggeration factor.

In [ ]:
vertical_exaggeration = 40.0
topo_units = 'm'  # ETOPO data is in meters

scale = calculate_displacement_scale(
    model_radius_mm=model_radius_mm,
    vertical_exagg=vertical_exaggeration,
    grid_units=topo_units,
)
print(f"Scale factor: {scale}")

## Step 5: Apply Topography Displacement

We now translate each vertex on our sphere inward or outward along its normal vector according to the scaled elevation grid, creating mountains and trenches.

In [ ]:
# Apply displacement to the outer shell using a Displacer object
model.outer.displace(GridDisplacer(grid_ds, show_progress=True), scale=scale)
print("Displaced model vertices successfully.")

## Step 6: Preview Your Globe in 3D

Let's visualize the displaced vertices in 3D using `matplotlib` to make sure the features look correct before exporting.

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot a random subset of 1,000 points to keep plotting fast
indices = np.random.choice(len(model.outer.vertices), 1000, replace=False)
pts = model.outer.vertices[indices]
sc = ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=pts[:, 2], cmap='terrain', s=4)
fig.colorbar(sc, ax=ax, label='Z coordinate (mm)')
ax.set_title("Basic Globe Topography (3D Preview)")
plt.show()

## Step 7: Export to Binary STL

Finally, we save the displaced outer shell mesh to an STL file, ready to import into your 3D printer slicing software (like Bambu Studio, OrcaSlicer, or Cura).

In [ ]:
output_path = "../outputs/example_1_basic_globe.stl"

model.export(output_path)
print(f"Saved STL file to: {output_path}")